# [LandShift] Ecosystem extent

---
<div>The aim of this notebooks is to compute the ecosystem extent based on MAES taxonomy and tacking as input and LCCS map</div>
<div>More details on the methodology are available in Deliverable D3.5 </div>

<br><i>Author(s):</i> <a href='https://www.unige.ch/envirospace/people/giuliani' target='_blank'>Gregory Giuliani</a>
<br><i>Version:</i> 1.0
<br><i>Date:</i> 2026-04-21
<br><i>Supported by:</i> Horizon-Europe <a href='https://landshift.eu' target='_blank'>LandShift</a> project

---
### Load librairies

In [1]:
import os
import rasterio
import numpy as np
from rasterio.crs import CRS

### File

In [2]:
lccsFile = '/Users/gregorygiuliani/Desktop/ecosystem/input/LE_D26_basilicata_2020_data.tif' #TBC
outputFld = '/Users/gregorygiuliani/Desktop/ecosystem/output/'

In [3]:
# Open the raster file
with rasterio.open(lccsFile) as src:
    band_names = src.descriptions

print('Bands: '+str(band_names))
print('CRS: '+str(src.crs))

Bands: (None, None, None, None, None, None, None, None)
CRS: EPSG:6875


## Ecosystems - MAES
https://www.eea.europa.eu/en/analysis/indicators/ecosystem-coverage-in-europe

In [6]:
#Level 2
#output
name = os.path.basename(lccsFile)[:-len("_data.tif")]
output_raster = outputFld+name+'_maesL2.tif'

# Open the multiband raster
with rasterio.open(lccsFile) as src:
    # Read all bands as a 3D numpy array: (bands, rows, cols)
    data = src.read()
    profile = src.profile.copy()

# Extract individual bands for readability
level3 = data[0]
lifeform = data[1]
canopycover = data[2]
canopyheight = data[3]
leaftype = data[4]
croptype = data[5]

# Initialize classification array and assign the default value 0
classification = np.full(level3.shape, 0, dtype=np.uint8)

# Apply your conditions
classification[(level3 == 215)] = 101 #Urban (Settlements and other artificial areas)
classification[(level3 == 111)] = 102 #Cropland
classification[(level3 == 123)] = 102 #Cropland
classification[(level3 == 112) & (lifeform == 2)] = 103 #Grassland
classification[(level3 == 112) & (lifeform == 1) & (canopyheight <= 7)] = 104 #Forest and woodlands
classification[(level3 == 112) & (lifeform == 1) & (canopyheight >= 8)] = 105 #Heathland and shrub
classification[(level3 == 216)] = 106 #Sparsely vegetated land
classification[(level3 == 124)] = 107 #Wetlands
classification[(level3 == 220)] = 200 #Rivers and lakes
#301 - Marine inlets and transitional waters
#302 - Coastal
#303 - Shelf
#304 - Open Ocean

# Update the profile for a single-band output
profile.update(count=1, dtype=rasterio.uint8, crs=src.crs)

# Write the new classified raster
with rasterio.open(output_raster, 'w', **profile) as dst:
    dst.write(classification, 1)

print("✅ Classification complete and saved to", output_raster)


✅ Classification complete and saved to /Users/gregorygiuliani/Desktop/ecosystem/output/LE_D26_basilicata_2020_maesL2.tif


In [7]:
#Level 1
#input
input_raster = outputFld+name+'_maesL2.tif'
#output
output_raster = outputFld+name+'_maesL1.tif'

#Reclassify
with rasterio.open(input_raster) as src:
    # Read as numpy array
    array = src.read()
    profile = src.profile

    # Reclassify
    array[np.where(array == 101)] = 10 #Terrestrial
    array[np.where(array == 102)] = 10
    array[np.where(array == 103)] = 10
    array[np.where(array == 104)] = 10
    array[np.where(array == 105)] = 10
    array[np.where(array == 106)] = 10
    array[np.where(array == 107)] = 10
    array[np.where(array == 200)] = 20 #Freshwater
    array[np.where(array == 301)] = 30 #Marine
    array[np.where(array == 302)] = 30
    array[np.where(array == 303)] = 30
    array[np.where(array == 304)] = 30

with rasterio.open(output_raster, 'w', **profile) as dst:
    # Write to disk
    dst.write(array)

print(f"Reclassified file '{output_raster}' created successfully.")

Reclassified file '/Users/gregorygiuliani/Desktop/ecosystem/output/LE_D26_basilicata_2020_maesL1.tif' created successfully.


In [8]:
# Compute area in hectares per MAES class (Level 2 and Level 1) and export to CSV
try:
  import pandas
except:
  !pip install pandas
  import pandas as pd

import math

# MAES class labels
maes_l2_labels = {
    0:   'No data',
    101: 'Urban (Settlements and other artificial areas)',
    102: 'Cropland',
    103: 'Grassland',
    104: 'Forest and woodlands',
    105: 'Heathland and shrub',
    106: 'Sparsely vegetated land',
    107: 'Wetlands',
    200: 'Rivers and lakes',
}

maes_l1_labels = {
    0:  'No data',
    10: 'Terrestrial',
    20: 'Freshwater',
    30: 'Marine',
}

def compute_area_ha(raster_path, class_labels, export_csv=True, csv_path=None):
    """
    Compute pixel count and area (ha) for each class in a classified raster.

    Parameters
    ----------
    raster_path : str   Path to the input single-band raster.
    class_labels: dict  Mapping of pixel value -> human-readable label.
    export_csv  : bool  If True, save results to a CSV file (default: True).
    csv_path    : str   Output CSV path. Defaults to raster_path with .csv extension.

    Returns
    -------
    pd.DataFrame with columns: class_value, label, pixel_count, area_ha
    """
    with rasterio.open(raster_path) as src:
        data      = src.read(1)
        transform = src.transform
        crs       = src.crs

    # Pixel size in map units
    pixel_width  = abs(transform.a)
    pixel_height = abs(transform.e)

    if crs.is_geographic:
        lat_rad = math.radians(np.mean([transform.f, transform.f + transform.e * data.shape[0]]))
        pixel_area_m2 = (pixel_width * 111320 * math.cos(lat_rad)) * (pixel_height * 110540)
        print('⚠️  CRS is geographic – area is approximate.')
    else:
        pixel_area_m2 = pixel_width * pixel_height

    pixel_area_ha = pixel_area_m2 / 10_000  # 1 ha = 10 000 m²

    print(f'Pixel size : {pixel_width:.2f} x {pixel_height:.2f} m  →  {pixel_area_ha:.6f} ha/pixel')
    print(f'CRS        : {crs}\n')

    unique_vals, counts = np.unique(data, return_counts=True)

    # Build DataFrame
    rows = []
    for val, count in zip(unique_vals, counts):
        rows.append({
            'class_value': int(val),
            'label':       class_labels.get(int(val), f'Unknown ({val})'),
            'pixel_count': int(count),
            'area_ha':     round(count * pixel_area_ha, 2),
        })
    df = pandas.DataFrame(rows)

    # Print table
    print(f'{"Class value":>15}  {"Label":<48}  {"Pixels":>12}  {"Area (ha)":>14}')
    print('-' * 95)
    for _, row in df.iterrows():
        print(f'{row.class_value:>15}  {row.label:<48}  {row.pixel_count:>12,}  {row.area_ha:>14,.2f}')

    # Export to CSV
    if export_csv:
        if csv_path is None:
            csv_path = os.path.splitext(raster_path)[0] + '_area_ha.csv'
        df.to_csv(csv_path, index=False)
        print(f'\n✅ CSV saved to {csv_path}')

    return df

print('=' * 95)
print('MAES Level 2 – ecosystem extent')
print('=' * 95)
df_l2 = compute_area_ha(outputFld+name+'_maesL2.tif', maes_l2_labels)

print()
print('=' * 95)
print('MAES Level 1 – ecosystem extent')
print('=' * 95)
df_l1 = compute_area_ha(outputFld+name+'_maesL1.tif', maes_l1_labels)


MAES Level 2 – ecosystem extent
Pixel size : 10.00 x 10.00 m  →  0.010000 ha/pixel
CRS        : EPSG:6875

    Class value  Label                                                   Pixels       Area (ha)
-----------------------------------------------------------------------------------------------
              0  No data                                             82,416,057      824,160.57
            101  Urban (Settlements and other artificial areas)       3,376,490       33,764.90
            102  Cropland                                            30,920,449      309,204.49
            103  Grassland                                           21,089,434      210,894.34
            104  Forest and woodlands                                47,660,566      476,605.66
            105  Heathland and shrub                                     71,737          717.37
            106  Sparsely vegetated land                              2,978,430       29,784.30
            107  Wetlands    